In [0]:
# read data from raw_data volume - done
# add audit columnms -> 2 columns - last_update_ts, file_path
# define target
# write data in delta table

import pyspark.sql.functions as F
from common_utils.logging import get_logger
logger=get_logger('ds2b_sqlserver')

logger.info("Defining raw path")
raw_path='/Volumes/retaildataplatform/bronze/raw_data/sales/load_date=2026-09-19/'

logger.info("raw path given is %s",raw_path)

logger.info("reading data")
df=spark.read.format('parquet')\
    .option('header',"true")\
    .option('inferSchema','true')\
    .option('quoteAll','true')\
    .option('escape','true')\
    .load(raw_path)

logger.info("sample data %s",df.show())
logger.info("adding audit columns")
df=df.withColumn('last_updated_ts',F.current_timestamp())\
    .withColumn('file_path',F.col('_metadata.file_path'))

target_path='retaildataplatform.bronze.s3Sales_customers'

logger.info("target path is %s",target_path)
df=df.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable(target_path)
logger.info("Data writeen Successfully at %s", target_path)


